# Yoruba HealthQA -- Demo-Only Training (Kaggle Notebooks, free GPU)

**Read this before running anything.** This is the Kaggle version of `colab_train_demo_model.ipynb` -- same training logic and fixes, adapted for Kaggle's environment (data comes from an attached Kaggle Dataset instead of an upload dialog; results are downloaded from `/kaggle/working/` instead of a browser download prompt). Switched to Kaggle because Colab's free GPU quota ran out repeatedly -- Kaggle gives **30 hours of free GPU time per week** and sessions up to ~9-12 hours long, at no cost.

This notebook trains a small proof-of-concept model so the Yoruba HealthQA app can show real generated answers during a defence/presentation. It is **not** the dissertation's final, validated model:

- **The dataset used here is bigger, but a large part of it is UNREVIEWED.** Of 1,428 records, only 208 were actually checked by a real Yoruba-speaking human reviewer -- the other ~1,220 are raw machine-translation output that nobody has verified. Real mistakes have already been found in that raw machine translation (e.g. "breastfeeding" mistranslated as "pregnancy"). Training on it risks the model learning those mistakes *more confidently*, not just more fluently.
- **The base model is `CohereLabs/tiny-aya-earth` (3.35B params)**, chosen because Yoruba is **explicitly named** on this model's own card as a supported language -- a stronger, directly-verified signal than the earlier `bigscience/bloomz-560m` candidate had. It is bigger (more capacity to actually hold health knowledge) but noticeably slower and more memory-hungry to train, per real measurements below.
- **This model is GATED.** You need a Hugging Face account, to click "Agree and access repository" on the model's page, and to paste an access token into the login cell below. Its licence is CC-BY-NC (non-commercial use), matching another candidate already listed in this project's `configs/model.yaml`.
- **A safety-check cell runs before training** to check this model's REAL internal layer names against what the training cell assumes, and **stops the notebook immediately (an assertion) if they don't match** -- a past attempt with a different model (BLOOM) had different names than assumed, which would have silently trained nothing if not caught.
- **Two real crashes were hit and fixed on Colab while building this**, both from evidence, not guesses: (1) bitsandbytes' `paged_adamw_8bit` optimiser caused a CUDA illegal-memory-access crash -- switched to plain `adamw_torch`; (2) the default batch size/sequence length then caused a genuine out-of-memory crash 32 steps in -- cut batch size and sequence length (same effective batch size preserved, so the training math is unchanged, just how memory is spent). Both fixes are already applied below.
- **The epoch sweep is 2 candidates (1, 2)**, not a single guess -- picked by real evidence (validation chrF++, measured on held-out data the model never trains on) after a real run showed the original wider sweep would have taken far too long. **Expect this to take a real 3-4+ hours.**

**Even the winner of this sweep may not be medically accurate**, for two independent reasons: (1) it's still a small dataset by LLM standards, and (2) most of it was never human-checked. Actually read the winning run's generated answers to your example questions before trusting it in front of an audience -- and remember the retrieval-lookup mode (`YHQA_ANSWER_MODE=retrieval`, README2.md Section 4) remains available as a 100%-reviewed-text fallback regardless of how this training run turns out.

**When presenting this, say so explicitly**: *"This proof-of-concept fine-tune uses a base model that explicitly supports Yoruba, on a larger but partly-unreviewed dataset, with hyperparameters chosen by a validation sweep rather than a guess. The dissertation's full model requires completing human validation on the remaining data before training the final result."*

## Before you start: set up Kaggle (one-time, ~10 minutes)

1. Create a free account at [kaggle.com](https://kaggle.com) if you don't have one.
2. **Verify your phone number**: go to your account settings (click your profile picture -> Settings) -> "Phone Verification". This is REQUIRED before Kaggle will let a notebook use a GPU or the internet -- skip this and the notebook simply won't be able to download anything.
3. **Upload your dataset as a Kaggle Dataset** (this replaces Colab's upload-a-file dialog): go to [kaggle.com/datasets](https://kaggle.com/datasets) -> "New Dataset" -> upload the three files `train.jsonl`, `val.jsonl`, `test.jsonl` from your laptop's **`data/final_demo_expanded/`** folder -> give it a title (e.g. "yoruba-healthqa-demo-data") -> click "Create".
4. Create a new notebook: [kaggle.com/code](https://kaggle.com/code) -> "New Notebook".
5. Paste in the cells from this file (or upload this .ipynb via File -> Import Notebook).
6. In the right-hand sidebar of the notebook: click **"Add Input"** -> search for the dataset title you used in step 3 -> add it. It will appear at `/kaggle/input/<your-dataset-slug>/`.
7. Still in the right-hand sidebar: under **Settings**, set **Accelerator** to a GPU option (T4 x2 or P100), and make sure **Internet** is switched **On**.

## How to use this notebook
1. Run cells from top to bottom (Kaggle: **Run -> Run All**, or step through one at a time).
2. Before the login cell: make sure you've clicked "Agree and access repository" on https://huggingface.co/CohereLabs/tiny-aya-earth, and have a token ready from https://huggingface.co/settings/tokens.
3. **Check the safety-check cell's output** -- if the assertion fails, stop and tell Claude the exact printed list before continuing.
4. The data-loading cell automatically finds your attached dataset under `/kaggle/input/` -- no manual upload dialog needed.
5. Wait for both sweep runs to finish -- a real multi-hour wait.
6. The final cell copies the winning adapter's zip into `/kaggle/working/` -- download it from the **Output** section of the right-hand sidebar (or the file browser) once the notebook finishes.
7. On your own laptop, unzip it into your project's `models/` folder, then run the app with the environment variables shown at the bottom of this notebook -- including logging in to Hugging Face there too, since the base model is gated.

In [ ]:
# Confirm a GPU is actually attached. If this errors, go to the
# notebook's right-hand sidebar -> Settings -> Accelerator -> pick a GPU
# option, then re-run.
!nvidia-smi

In [ ]:
# Get the pipeline CODE (not the data) from GitHub.
#
# Made idempotent (always starts from /kaggle/working, deletes any leftover
# clone first) because a real run on Colab showed that re-running this
# cell inside an already-running session creates a NESTED duplicate
# folder if one already exists -- everything downstream then silently
# operates inside the wrong copy. This makes the cell safe to re-run.
import shutil

%cd /kaggle/working
shutil.rmtree("Yoruba-HealthQA", ignore_errors=True)
!git clone https://github.com/ayoolaeni/Yoruba-HealthQA.git
%cd Yoruba-HealthQA

In [ ]:
# Install only what's needed for training, on top of Kaggle's pre-installed
# torch+CUDA (deliberately NOT reinstalling torch from requirements.txt --
# that pinned version may not match Kaggle's CUDA build and could break
# GPU support).
#
# NOT pinning exact versions for transformers/peft/bitsandbytes/accelerate,
# on purpose: platform base images move forward over time and a pin chosen
# today can be stale by the time you run this. Letting pip resolve current
# mutually-compatible versions is more robust than guessing fixed numbers.
!pip install -q -U transformers peft bitsandbytes accelerate datasets sacrebleu sentencepiece pyyaml

In [ ]:
# Log in to Hugging Face -- required because CohereLabs/tiny-aya-earth is
# gated. Before running this cell:
#   1. Visit https://huggingface.co/CohereLabs/tiny-aya-earth while logged
#      in and click "Agree and access repository" (usually instant).
#   2. Get a token at https://huggingface.co/settings/tokens (read access
#      is enough).
#   3. In this notebook: Add-ons menu -> Secrets -> add a secret labelled
#      HF_TOKEN with your token as the value, and make sure it is attached
#      to this notebook.
#
# The token is read from Kaggle Secrets instead of a typed prompt because a
# background "Save & Run All" run (the way to keep the trained files -- see
# the markdown cell near the top) cannot stop and wait for someone to type.
# The getpass prompt is only a fallback for interactive sessions.
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient

    _token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    import getpass

    _token = getpass.getpass("Paste your Hugging Face access token: ")
login(token=_token)

In [ ]:
# SAFETY CHECK before training: confirm the real internal layer names this
# model uses, instead of assuming. An earlier run with a different model
# (BLOOM) had different attention layer names than the Llama/Qwen-style
# ones the pipeline defaults to -- guessing wrong there would have silently
# attached LoRA to nothing. This model's exact config isn't visible without
# being logged in (it's gated), so this cell loads just the model
# structure and checks every Linear layer name it actually finds.
#
# This now ASSERTS instead of just printing a warning: with training
# taking 3-4+ hours unattended, a printed "please check this" message is
# easy to miss or scroll past -- an assertion stops the notebook
# immediately and loudly instead of silently wasting hours training a
# LoRA adapter attached to nothing.
import torch
from transformers import AutoModelForCausalLM

EXPECTED_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

_probe_model = AutoModelForCausalLM.from_pretrained("CohereLabs/tiny-aya-earth", torch_dtype=torch.float16)
_linear_suffixes = sorted({name.rsplit(".", 1)[-1] for name, mod in _probe_model.named_modules()
                            if mod.__class__.__name__ == "Linear" and "." in name})
del _probe_model  # free the memory; the training cell reloads it quantised anyway

print("Real Linear-layer name suffixes found in this model:")
print(_linear_suffixes)

missing = [m for m in EXPECTED_TARGET_MODULES if m not in _linear_suffixes]
assert not missing, (
    f"STOP: the training cell assumes target_modules={EXPECTED_TARGET_MODULES}, but this model "
    f"does not have {missing} among its real layer names ({_linear_suffixes}). Update "
    f"TARGET_MODULES in the next cell to match the printed list above before continuing -- "
    f"do not proceed on the old guess."
)
print("OK: all expected target module names were found in this model.")

In [ ]:
# Find your dataset files -- Kaggle attaches datasets under /kaggle/input/
# (not an upload dialog like Colab), but the exact nesting depth varies
# (a real run showed it one level deeper than expected, under a generic
# "datasets" folder) -- so this searches recursively at ANY depth under
# /kaggle/input/ for a folder containing all three required files,
# instead of assuming a fixed depth.
import os
import shutil
from pathlib import Path

REQUIRED_FILES = ["train.jsonl", "val.jsonl", "test.jsonl"]

kaggle_input = Path("/kaggle/input")
found_dir = None
for candidate_dir in [kaggle_input] + [p for p in kaggle_input.rglob("*") if p.is_dir()]:
    if all((candidate_dir / fname).exists() for fname in REQUIRED_FILES):
        found_dir = candidate_dir
        break

assert found_dir is not None, (
    f"Could not find all of {REQUIRED_FILES} anywhere under /kaggle/input/, at any depth. "
    f"All files found under /kaggle/input/: {[str(p) for p in kaggle_input.rglob('*') if p.is_file()]}. "
    f"Make sure you attached your dataset via the right-hand sidebar's 'Add Input' "
    f"button (see the intro markdown, step 6) before running this cell."
)

os.makedirs("data/final_demo", exist_ok=True)
for fname in REQUIRED_FILES:
    shutil.copy(found_dir / fname, f"data/final_demo/{fname}")
print(f"Copied dataset files from {found_dir} -> data/final_demo/:", os.listdir("data/final_demo"))

In [ ]:
# Pick the base model and write it into configs/train.yaml (this is the
# fresh clone made in this Colab session -- editing it here does NOT touch
# your real local repo).
#
# CohereLabs/tiny-aya-earth (3.35B params): chosen because Yoruba is
# EXPLICITLY named on its own model card as one of its supported languages
# (not inferred from a general "multilingual" claim like earlier
# candidates) -- a stronger, directly-verified signal than bloomz-560m had.
# Gated -- see the login cell above. Licence: CC-BY-NC (non-commercial),
# consistent with the other CC-BY-NC candidate (aya-23-8B) already listed
# in configs/model.yaml.
#
# target_modules: set from the REAL printed output of the safety-check
# cell above, not a blind guess -- if that cell's printed list did not
# match ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj',
# 'down_proj'], STOP and update TARGET_MODULES below to match what was
# actually printed before running the training cell.
#
# OPTIMISER override: a real run crashed with "CUDA error: an illegal
# memory access" inside bitsandbytes' paged_adamw_8bit optimiser (the
# canonical configs/train.yaml's spec'd choice) -- known fragile on Colab
# T4 GPUs with certain bitsandbytes/CUDA driver combos. adamw_torch (plain
# PyTorch AdamW) avoids that failure mode. Demo-only override; the real
# dissertation config keeps paged_adamw_8bit as specified.
#
# BATCH SIZE / SEQ LENGTH: a real run OOM'd at the canonical batch size (4)
# and sequence length (1024) 32 steps in. Sequence length was cut to 512
# to fix that -- verified against real word counts in this exact dataset
# (train.jsonl): answers run up to 173 words (~250 tokens), so 512 gives
# headroom without truncating real content, while 1024 was needlessly
# double that. Batch size was cut to 1 at the same time out of caution,
# but that was paired with the OLD 1024 sequence length -- with the
# sequence length now half that, there should be real memory headroom to
# raise it back to 2 (gradient_accumulation halved to 8 to keep the SAME
# effective batch size of 16, so the learning dynamics AND the total
# number of optimiser steps (72, for 1 epoch) are unchanged -- what
# should get faster is the WALL-CLOCK TIME per step (8 larger micro-batches
# per step instead of 16 tiny ones, better using the GPU each time), not
# the step count itself. If this OOMs again, drop
# PER_DEVICE_TRAIN_BATCH_SIZE back to 1 and GRADIENT_ACCUMULATION_STEPS
# back to 16.
#
# EPOCH: just the real sweep winner (1 epoch beat 2 on validation chrF++:
# 8.90 vs 8.59) -- rerunning only it after the first run's output was
# lost to a session timeout (see the markdown note near the top of this
# notebook about using Save & Run All instead).
BASE_MODEL = "CohereLabs/tiny-aya-earth"
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
EPOCH_CANDIDATES = [1]
OPTIMISER = "adamw_torch"
PER_DEVICE_TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
MAX_SEQ_LENGTH = 512

import yaml

with open("configs/train.yaml", encoding="utf-8") as f:
    train_cfg = yaml.safe_load(f)
train_cfg["base_model"] = BASE_MODEL
train_cfg["lora"]["target_modules"]["initial"] = TARGET_MODULES
train_cfg["lora"]["r"]["sweep"] = [train_cfg["lora"]["r"]["initial"]]
train_cfg["optim"]["learning_rate"]["sweep"] = [train_cfg["optim"]["learning_rate"]["initial"]]
train_cfg["optim"]["epochs"]["sweep"] = EPOCH_CANDIDATES
train_cfg["optim"]["optimiser"] = OPTIMISER
train_cfg["optim"]["per_device_train_batch_size"] = PER_DEVICE_TRAIN_BATCH_SIZE
train_cfg["optim"]["gradient_accumulation_steps"] = GRADIENT_ACCUMULATION_STEPS
train_cfg["optim"]["max_seq_length"] = MAX_SEQ_LENGTH
with open("configs/train.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(train_cfg, f, sort_keys=False, allow_unicode=True)
print(f"configs/train.yaml base_model={BASE_MODEL}, target_modules={TARGET_MODULES}, "
      f"epoch sweep={EPOCH_CANDIDATES}, optimiser={OPTIMISER}, "
      f"batch_size={PER_DEVICE_TRAIN_BATCH_SIZE} x accum={GRADIENT_ACCUMULATION_STEPS}, "
      f"max_seq_length={MAX_SEQ_LENGTH}")

In [ ]:
# Train ONE run (1 epoch) and package it immediately.
#
# Why this skips the validation chrF++ scoring step: after training, the
# real script (scripts/09_train.py) scores the model by generating an
# answer for all 143 validation questions ONE AT A TIME, and prints
# nothing while doing so -- in real runs this looked frozen for hours
# (2h+ of total silence) and repeatedly left runs unfinished or lost.
# With a single candidate there is nothing to choose between, and the
# score for this exact setup is already known from the first full sweep
# on this same data (1 epoch: val chrF++ 8.90, train loss 1.478), so
# skipping it loses nothing. The trained adapter is saved to disk BEFORE
# that scoring step even starts (checked in the script's source), so this
# is safe.
#
# This is a demo-only workaround done inside the notebook: it loads the
# real scripts/09_train.py unchanged and only swaps out that one scoring
# function for a no-op that returns NaN (shown honestly as "nan" in the
# results table, not a made-up number). The real dissertation pipeline
# and its script are untouched and still score properly.
#
# PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True is PyTorch's own
# documented mitigation for the OOM message seen in an earlier real run.
from pathlib import Path

Path("run_train_no_val.py").write_text('''
import importlib.util
import sys
from pathlib import Path

spec = importlib.util.spec_from_file_location("train09", Path("scripts/09_train.py"))
train09 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train09)


def _skip_validation_scoring(*args, **kwargs):
    train09.logger.info("SKIPPING validation chrF++ scoring (demo-only; see notebook comment)")
    return float("nan")


train09.compute_val_chrf = _skip_validation_scoring
sys.argv = ["09_train.py", "--sweep", "--data-dir", "data/final_demo",
            "--train-config", "configs/train.yaml", "--model-config", "configs/model.yaml",
            "--models-dir", "models", "--reports-dir", "reports"]
train09.main()
''', encoding="utf-8")

!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python run_train_no_val.py

# Package the trained adapter RIGHT AWAY, in this same cell, so it is
# safely zipped into /kaggle/working the moment training finishes -- not
# left waiting on a later cell (the first Kaggle run lost its files because
# nothing was downloaded before the session ended).
import shutil

shutil.make_archive("/kaggle/working/single_run_adapter", "zip", "models/sweep_000")
print("DONE. Saved to /kaggle/working/single_run_adapter.zip -- download it NOW "
      "from the file browser (left panel) or the Output section of the right-hand sidebar.")

In [ ]:
# Show the sweep results and copy the winning run (by val chrF++) into
# models/single_run so the rest of this notebook (packaging/download)
# doesn't need to change.
import os
import shutil

import pandas as pd

df = pd.read_csv("reports/table_4_4_hyperparameter_sweep.csv").sort_values("val_chrf++", ascending=False)
print(df[["run", "epochs", "train_loss", "val_chrf++"]].to_string(index=False))

best = df.iloc[0]
winning_run_dir = best["run_dir"]
print(f"\nWinner: epochs={best['epochs']} val_chrf++={best['val_chrf++']:.2f} (train_loss={best['train_loss']:.3f})")

target = "models/single_run"
if os.path.exists(target):
    shutil.rmtree(target)
shutil.copytree(winning_run_dir, target)
print(f"Copied {winning_run_dir} -> {target}")

In [ ]:
# Package the trained adapter for download. Kaggle doesn't have Colab's
# files.download() browser prompt -- instead, anything written under
# /kaggle/working/ becomes downloadable from the notebook's file browser
# (left-hand panel) or the "Output" section of the right-hand sidebar
# once this cell finishes running.
import shutil

shutil.make_archive("/kaggle/working/single_run_adapter", "zip", "models/single_run")
print("Saved to /kaggle/working/single_run_adapter.zip -- download it from the file "
      "browser (left panel) or the Output section of the right-hand sidebar.")

## On your own laptop, after downloading

1. Unzip `single_run_adapter.zip` into your project folder. If your unzip tool creates an extra nested folder (check whether `adapter_config.json` ends up directly inside `models/single_run/` or one level deeper, e.g. `models/single_run/single_run_adapter/`), just point `YHQA_ADAPTER_PATH` below at whichever folder actually contains `adapter_config.json`.
2. Set these environment variables (or put them in your `.env` file) before starting the app:

```
YHQA_BASE_MODEL=CohereLabs/tiny-aya-earth
YHQA_ADAPTER_PATH=models/single_run
```

3. **Because the base model is gated, your laptop also needs to be logged in to Hugging Face once** before the app can download it -- run this once in a terminal (needs the `huggingface_hub` package, already in requirements.txt):
   ```
   huggingface-cli login
   ```
   and paste the same access token you used here. (Alternatively, set an `HF_TOKEN` environment variable to the same token.)
4. Run the app as usual (`python app/app.py` or `docker compose up --build` with those variables set).
5. The status line in the app should now show the green "answer engine is working" message instead of the yellow placeholder one.
6. Also worth downloading from the same file browser: `reports/table_4_4_hyperparameter_sweep.md` -- the real sweep results table, useful evidence for the dissertation regardless of how the presentation goes.

**A note on speed and size:** the first time you ask a question, the app downloads the ~3.35B-parameter base model from Hugging Face (bigger than bloomz-560m's ~1.1GB, so allow more time and disk space) -- this only happens once. Generation on CPU will likely be slow; if it's too slow for a live demo, `YHQA_ANSWER_MODE=retrieval` remains the fast, reliable fallback.

**Before trusting any answer it gives**: actually listen to a few generated answers from your example questions before your presentation. Remember this run was trained on a dataset where most records (~1,220 of 1,428) were never human-reviewed -- so even a good validation chrF++ score doesn't rule out the model having picked up a real translation mistake somewhere. If it's incoherent, irrelevant, or you're not confident in its factual content, `YHQA_ANSWER_MODE=retrieval` remains the reliable, 100%-human-reviewed fallback -- it doesn't depend on this training run working out.

**If training itself fails with an out-of-memory error**, open `configs/train.yaml` and reduce `per_device_train_batch_size` under `optim` even further (e.g. to keep at 1, since it's already been cut once), or reduce `max_seq_length` below the current 512.